# 02 — Feature Engineering & Preprocessing Pipeline
## Telco Customer Churn — Customer Churn Intelligence System

**Notebook purpose:** Convert the raw, validated dataset from `01_data_quality_and_eda.ipynb` into a leakage-free, reusable preprocessing pipeline. The artifacts produced here (`preprocessor.pkl`, `feature_columns.json`) are **first-class production deliverables** — they will be loaded directly inside the FastAPI inference service and the batch scoring pipeline, so every transformation is implemented via `scikit-learn` transformers rather than ad-hoc pandas code, ensuring identical behavior between training and serving.

---

### Design Principles for This Notebook

1. **No transformation may be fit on data the model will later be evaluated on.** All fitting happens strictly on the training split.
2. **Every transformation must be serializable.** Hand-written pandas logic (e.g., `df['x'].fillna(...)`) cannot be shipped to a FastAPI service safely — it must live inside a `ColumnTransformer`/`Pipeline` object that is pickled once and loaded everywhere.
3. **The pipeline's input contract must match what the API will receive**: raw, unprocessed customer records. The pipeline itself is responsible for cleaning, encoding, and scaling.

## 1. Load Validated Data

In [1]:
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

RAW_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODELS_DIR = '../models'
CONFIGS_DIR = '../configs'

import os
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head(3)

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 2. Identifier Columns

Columns that carry no predictive signal and must never enter the model must be identified and dropped explicitly, rather than silently excluded, so the decision is auditable.

In [2]:
identifier_cols = ['customerID']
print(f"Identifier columns removed from the feature space: {identifier_cols}")

df = df.drop(columns=identifier_cols)
df.shape

Identifier columns removed from the feature space: ['customerID']


(7043, 20)

**Note:** `customerID` is dropped from the *modeling* feature set here, but it will still be needed downstream (e.g., batch inference output joins, CRM lookups). In production, the FastAPI/batch service will carry `customerID` alongside the record and simply exclude it before calling `pipeline.transform()` — it is not deleted from the source data, only from the model's feature space.

## 3. Handle `TotalCharges` Data Type

As identified in the EDA notebook, `TotalCharges` is stored as a string with 11 blank entries corresponding to customers with `tenure == 0`. This must be corrected **before** the train/test split so downstream type inference is consistent, but the *value imputation* itself (filling with 0) is safe to do here since it is a deterministic, data-independent business rule — not a statistic learned from the training set.

In [3]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Deterministic business rule: zero-tenure customers have zero total charges
zero_tenure_mask = df['TotalCharges'].isnull()
print(f"Rows requiring the tenure==0 -> TotalCharges=0 rule: {zero_tenure_mask.sum()}")
assert (df.loc[zero_tenure_mask, 'tenure'] == 0).all(), "Unexpected non-zero tenure with missing TotalCharges"

df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
print("Remaining nulls in TotalCharges:", df['TotalCharges'].isnull().sum())
df['TotalCharges'].describe()

Rows requiring the tenure==0 -> TotalCharges=0 rule: 11
Remaining nulls in TotalCharges: 0


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64

## 4. Target Encoding

In [4]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts(normalize=True).round(3)

Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

## 5. Feature / Target Split and Train/Test Split

The split happens **before** any encoder or scaler is fit, so the pipeline never sees test data during `fit()`. Stratification preserves the ~27% churn rate in both splits.

In [5]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train shape: {X_train.shape}  |  churn rate: {y_train.mean():.3f}")
print(f"Test shape:  {X_test.shape}  |  churn rate: {y_test.mean():.3f}")

Train shape: (5634, 19)  |  churn rate: 0.265
Test shape:  (1409, 19)  |  churn rate: 0.265


## 6. Feature Typing Strategy

Features are grouped by the transformation they require, not just by pandas dtype. `SeniorCitizen` is stored as an integer 0/1 flag but is semantically categorical/binary, so it is grouped with the categorical features conceptually, while being left untouched numerically (it is already a clean binary encoding, so passing it through avoids unnecessary one-hot expansion).

In [6]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

binary_passthrough_features = ['SeniorCitizen']

categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

assert set(numerical_features + binary_passthrough_features + categorical_features) == set(X.columns), \
    "Feature groups do not cover all columns"

print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Binary passthrough features ({len(binary_passthrough_features)}): {binary_passthrough_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Numerical features (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Binary passthrough features (1): ['SeniorCitizen']
Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 7. Encoding Strategy

**One-Hot Encoding** is used for all categorical features rather than ordinal/label encoding, because none of these categories have a natural order (e.g., `InternetService` values `DSL`/`Fiber optic`/`No` are not ordinal), and tree-based models (Random Forest, XGBoost) as well as linear models (Logistic Regression) all handle one-hot features correctly, keeping a **single shared preprocessing pipeline** usable across every candidate model in the next notebook.

The redundant `"No internet service"` / `"No phone service"` categories identified in the EDA are **not manually collapsed**. One-hot encoding naturally absorbs them as their own indicator columns, and collapsing them would require custom, non-standard logic that adds fragility to the pipeline without a meaningful modeling benefit at this feature-set size. `handle_unknown='ignore'` is set so that any unseen category value at inference time (e.g., a new payment method added by the business later) does not crash the API — it is encoded as all-zeros for that feature instead of raising an exception.

## 8. Scaling Strategy

**StandardScaler** is applied to the numerical features (`tenure`, `MonthlyCharges`, `TotalCharges`). This is required for Logistic Regression to converge efficiently and to keep coefficients comparable, and is harmless (a no-op in effect) for the tree-based models evaluated later — so a single shared pipeline can serve all four model families without per-model preprocessing branches.

## 9. ColumnTransformer + Pipeline

In [7]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', binary_transformer, binary_passthrough_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

**Note:** `SimpleImputer` steps are included for all feature groups even though the current dataset has no residual missing values after the `TotalCharges` fix. This is a deliberate production-readiness decision: the batch inference pipeline will eventually ingest live, unvalidated data where missingness is possible, and a pipeline that assumes a perfectly clean input is not safe to deploy.

## 10. Fit Pipeline on Training Data Only

In [8]:
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out().tolist()

print(f"Transformed train shape: {X_train_transformed.shape}")
print(f"Transformed test shape:  {X_test_transformed.shape}")
print(f"Total output features: {len(feature_names_out)}")
print()
print("Sample of output feature names:")
print(feature_names_out[:10])

Transformed train shape: (5634, 45)
Transformed test shape:  (1409, 45)
Total output features: 45

Sample of output feature names:
['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No']


## 11. Feature Selection Discussion

No automated feature selection (e.g., recursive feature elimination, variance thresholding) is applied at this stage, for three reasons:

1. **Dimensionality is low.** After one-hot encoding, the feature space is on the order of 30–45 columns — far from the regime where tree-based models or regularized linear models struggle with the curse of dimensionality.
2. **Regularization handles it implicitly.** Logistic Regression uses L2 regularization by default, and both Random Forest and XGBoost perform implicit feature selection through split-importance during training. A separate pre-selection step would risk discarding a feature that a later model could have used effectively, without a corresponding accuracy benefit.
3. **Explainability requires the full feature set.** The SHAP analysis in `04_model_explainability.ipynb` is most informative when every original business feature (e.g., `Contract`, `PaymentMethod`) is retained and visible, rather than pruned away before the model ever sees them.

Feature importance **is** examined post-hoc in the modeling and explainability notebooks — this is feature selection performed *after* seeing what the model actually learned, which is more defensible than a blind pre-filter.

## 12. Persist Preprocessing Artifacts

These two artifacts are the contract between this notebook and the rest of the ML system (model training, FastAPI service, batch inference job).

In [9]:
preprocessor_path = f'{MODELS_DIR}/preprocessor.pkl'
joblib.dump(preprocessor, preprocessor_path)
print(f"Saved fitted preprocessing pipeline to: {preprocessor_path}")

feature_columns_payload = {
    'raw_input_columns': X.columns.tolist(),
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'binary_passthrough_features': binary_passthrough_features,
    'transformed_feature_names': feature_names_out,
    'n_output_features': len(feature_names_out),
    'target_column': 'Churn',
    'target_mapping': {'No': 0, 'Yes': 1},
    'identifier_columns_excluded': identifier_cols,
    'random_state': RANDOM_STATE,
    'test_size': 0.2
}

feature_columns_path = f'{CONFIGS_DIR}/feature_columns.json'
with open(feature_columns_path, 'w') as f:
    json.dump(feature_columns_payload, f, indent=2)
print(f"Saved feature metadata to: {feature_columns_path}")

Saved fitted preprocessing pipeline to: ../models/preprocessor.pkl
Saved feature metadata to: ../configs/feature_columns.json


## 13. Persist Train/Test Splits for the Next Notebook

Raw (untransformed) splits are saved so the modeling notebook can either reuse the already-fitted `preprocessor.pkl` directly, or refit within a full `Pipeline([...])` object per model — both patterns are demonstrated in `03_model_training.ipynb`.

In [10]:
X_train.to_csv(f'{MODELS_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{MODELS_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{MODELS_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{MODELS_DIR}/y_test.csv', index=False)

print("Persisted train/test splits to the models/ directory for 03_model_training.ipynb")
for fname in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']:
    print(f" - {MODELS_DIR}/{fname}")

Persisted train/test splits to the models/ directory for 03_model_training.ipynb
 - ../models/X_train.csv
 - ../models/X_test.csv
 - ../models/y_train.csv
 - ../models/y_test.csv


## 14. Sanity Check — Round-Trip Load

In [11]:
loaded_preprocessor = joblib.load(preprocessor_path)
sanity_transform = loaded_preprocessor.transform(X_test.head(5))
print("Round-trip transform shape:", sanity_transform.shape)
assert sanity_transform.shape[1] == len(feature_names_out)
print("Preprocessing artifact verified: loads correctly and produces the expected feature dimensionality.")

Round-trip transform shape: (5, 45)
Preprocessing artifact verified: loads correctly and produces the expected feature dimensionality.


## 15. Summary

- `TotalCharges` was corrected to a numeric type with a deterministic, auditable imputation rule (`tenure == 0 → 0.0`), applied identically to train and test.
- A single `ColumnTransformer` (median/most-frequent imputation, `StandardScaler` for numerics, `OneHotEncoder` for categoricals) now serves as the shared preprocessing contract for every model family evaluated next.
- The pipeline was fit **exclusively on the training split** to prevent leakage, and verified to transform unseen data correctly.
- Two production artifacts are now available in `models/` and `configs/`:
  - **`models/preprocessor.pkl`** — the fitted `sklearn` transformer, loadable directly by the FastAPI service and the batch scoring job.
  - **`configs/feature_columns.json`** — machine-readable metadata describing the exact input contract (raw columns expected) and output contract (transformed feature names), used to validate incoming API requests before they reach the model.

**Next:** `03_model_training.ipynb` trains and compares Logistic Regression, Decision Tree, Random Forest, and XGBoost on top of this pipeline.